# Archipelagic Waters Visualization with Folium

This notebook explores and visualizes archipelagic water regions from the World_Archipelagic_Waters_v4 GeoPackage using Folium interactive mapping.

**Dataset Information:**
- Source: Marine Regions (marineregions.org)
- Version: v4 (dated 20231025)
- License: CC-BY 4.0
- Data Type: Archipelagic Waters boundaries (GeoPackage format)
- Citation: Flanders Marine Institute (2023). Maritime Boundaries Geodatabase: Archipelagic Waters, version 4

## Section 1: Import Required Libraries

Import necessary libraries for data processing and visualization.

In [1]:
import geopandas as gpd
import folium
from folium import plugins
import os
import json
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## Section 2: Explore Directory Structure and File Contents

Display all files and directories in the Archipelagic Waters dataset folder.

In [2]:
# Define the dataset path
dataset_dir = "/home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/World_Boundaries/World_Archipelagic_Waters_v4_20231025_gpkg"

print("📁 Directory Structure:")
print("="*70)

# Use os.walk to explore the directory
for root, dirs, files in os.walk(dataset_dir):
    level = root.replace(dataset_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}📂 {os.path.basename(root)}/')
    
    # Print files in current directory
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        file_size = os.path.getsize(file_path) / (1024*1024)  # Convert to MB
        print(f'{subindent}📄 {file} ({file_size:.2f} MB)')

print("\n✓ Directory exploration complete")

📁 Directory Structure:
📂 World_Archipelagic_Waters_v4_20231025_gpkg/
  📄 eez_archipelagic_waters_v4.gpkg (38.11 MB)
  📄 LICENSE_ArchipelagicWaters_v4.txt (0.00 MB)

✓ Directory exploration complete


## Section 3: Load GeoPackage Data

Read the GeoPackage file and inspect available layers.

In [3]:
import fiona

# Path to GeoPackage file
gpkg_path = os.path.join(dataset_dir, "eez_archipelagic_waters_v4.gpkg")

print(f"📦 GeoPackage File: {os.path.basename(gpkg_path)}")
print(f"   File Size: {os.path.getsize(gpkg_path) / (1024*1024):.2f} MB")
print(f"   Location: {gpkg_path}\n")

# List layers in the GeoPackage
layers = fiona.listlayers(gpkg_path)
print(f"📊 Available Layers in GeoPackage:")
for i, layer in enumerate(layers, 1):
    print(f"   {i}. {layer}")

# Load the archipelagic waters layer
layer_name = layers[0]  # Usually the first layer
gdf = gpd.read_file(gpkg_path, layer=layer_name)

print(f"\n✓ Successfully loaded layer: {layer_name}")
print(f"   Shape: {gdf.shape[0]} features, {gdf.shape[1]} attributes")

📦 GeoPackage File: eez_archipelagic_waters_v4.gpkg
   File Size: 38.11 MB
   Location: /home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/World_Boundaries/World_Archipelagic_Waters_v4_20231025_gpkg/eez_archipelagic_waters_v4.gpkg

📊 Available Layers in GeoPackage:
   1. eez_archipelagic_waters_v4

✓ Successfully loaded layer: eez_archipelagic_waters_v4
   Shape: 23 features, 16 attributes


## Section 4: Inspect Data Schema and Geometry

Display detailed information about the geodata structure, columns, and spatial properties.

In [4]:
print("📋 GeoDataFrame Information:")
print("="*70)
print(f"Shape: {gdf.shape}")
print(f"Coordinate Reference System (CRS): {gdf.crs}")
print(f"Total Bounds: {gdf.total_bounds}")
print(f"\n🗂️ Columns:")
for i, col in enumerate(gdf.columns, 1):
    dtype = gdf[col].dtype
    null_count = gdf[col].isnull().sum()
    print(f"   {i:2d}. {col:20s} - {str(dtype):15s} (nulls: {null_count})")

print(f"\n📍 Geometry Types:")
print(f"   {gdf.geometry.type.value_counts().to_dict()}")

print(f"\n📊 Data Summary:")
print(gdf[['TERRITORY1', 'SOVEREIGN1', 'AREA_KM2']].describe())

print(f"\n📄 First 3 rows:")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(gdf[['GEONAME', 'TERRITORY1', 'SOVEREIGN1', 'AREA_KM2', 'POL_TYPE']].head(3))

📋 GeoDataFrame Information:
Shape: (23, 16)
Coordinate Reference System (CRS): EPSG:4326
Total Bounds: [-180.          -21.02833333  180.           27.27298372]

🗂️ Columns:
    1. MRGID                - int64           (nulls: 0)
    2. GEONAME              - object          (nulls: 0)
    3. POL_TYPE             - object          (nulls: 0)
    4. MRGID_TER1           - int64           (nulls: 0)
    5. TERRITORY1           - object          (nulls: 0)
    6. MRGID_SOV1           - int64           (nulls: 0)
    7. SOVEREIGN1           - object          (nulls: 0)
    8. ISO_TER1             - object          (nulls: 1)
    9. X_1                  - float64         (nulls: 0)
   10. Y_1                  - float64         (nulls: 0)
   11. MRGID_EEZ            - int64           (nulls: 0)
   12. AREA_KM2             - int64           (nulls: 0)
   13. ISO_SOV1             - object          (nulls: 0)
   14. UN_SOV1              - int64           (nulls: 0)
   15. UN_TER1              

,GEONAME,TERRITORY1,SOVEREIGN1,AREA_KM2,POL_TYPE
0,Vanuatuan Archipelagic Waters,Vanuatu,Vanuatu,70824,Archipelagic waters
1,Solomon Island Archipelagic Waters,Solomon Islands,Solomon Islands,129096,Archipelagic waters
2,Philippine Archipelagic Waters,Philippines,Philippines,591370,Archipelagic waters


## Section 5: Prepare Data for Visualization

Validate geometries, check CRS, and prepare the data for mapping.

In [5]:
# Check for null geometries
print("🔍 Data Validation:")
print("="*70)
null_geom = gdf.geometry.isnull().sum()
invalid_geom = (~gdf.geometry.is_valid).sum()
print(f"   Null geometries: {null_geom}")
print(f"   Invalid geometries: {invalid_geom}")
print(f"   Valid geometries: {len(gdf) - null_geom - invalid_geom}")

# Ensure WGS84 projection (EPSG:4326)
if gdf.crs is None:
    print("\n⚠️  CRS not defined, setting to WGS84 (EPSG:4326)")
    gdf = gdf.set_crs('EPSG:4326')
elif gdf.crs != 'EPSG:4326':
    print(f"\n🔄 Reprojecting from {gdf.crs} to EPSG:4326 (WGS84)...")
    gdf = gdf.to_crs('EPSG:4326')
    print("   ✓ Reprojection complete")
else:
    print(f"\n✓ Already in WGS84 (EPSG:4326)")

print(f"\n📊 Statistics:")
print(f"   Total area: {gdf['AREA_KM2'].sum():,.0f} km²")
print(f"   Number of territories: {gdf['TERRITORY1'].nunique()}")
print(f"   Number of sovereigns: {gdf['SOVEREIGN1'].nunique()}")

print(f"\n✓ Data preparation complete")

🔍 Data Validation:
   Null geometries: 0
   Invalid geometries: 0
   Valid geometries: 23

✓ Already in WGS84 (EPSG:4326)

📊 Statistics:
   Total area: 5,363,643 km²
   Number of territories: 23
   Number of sovereigns: 22

✓ Data preparation complete


## Section 6: Create Base Map with Folium

Initialize a Folium map centered on an appropriate location to display all archipelagic waters.

In [6]:
# Get bounds of the data
bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

print(f"🗺️  Map Center: ({center_lat:.2f}, {center_lon:.2f})")
print(f"   Bounds: {bounds}")

# Create base map
map_archipelagic = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=2,
    tiles='OpenStreetMap',
    max_bounds=True
)

# Add title to the map
title_html = '''
             <div style="position: fixed; 
                     top: 10px; left: 50px; width: 400px; height: 60px; 
                     background-color: white; border:2px solid grey; z-index:9999; 
                     font-size:16px; font-weight: bold; padding: 10px;
                     border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
             🌍 Archipelagic Waters Visualization
             <br><small>Marine Regions - v4 (2023)</small>
             </div>
             '''
map_archipelagic.get_root().html.add_child(folium.Element(title_html))

print("✓ Base map created successfully")

🗺️  Map Center: (3.12, 0.00)
   Bounds: [-180.          -21.02833333  180.           27.27298372]
✓ Base map created successfully


## Section 7: Add Archipelagic Waters Layer to Map

Convert GeoDataFrame to GeoJSON and add as an interactive layer with custom styling.

In [7]:
# Convert to GeoJSON
geojson_data = gdf.to_json()

# Define colors for different territories
territory_colors = {
    territory: f'#{hash(territory) & 0xFFFFFF:06x}' 
    for territory in gdf['TERRITORY1'].unique()
}

print(f"🎨 Territory Colors:")
for territory, color in sorted(territory_colors.items())[:5]:
    print(f"   {territory:30s} -> {color}")

# Add GeoJSON layer with custom styling
def style_function(feature):
    territory = feature['properties'].get('TERRITORY1', 'Unknown')
    return {
        'fillColor': '#1f77b4',  # Blue color
        'color': '#003d7a',      # Darker blue border
        'weight': 2,
        'opacity': 0.8,
        'fillOpacity': 0.4,
        'dashArray': '5, 5'
    }

# Add the GeoJSON layer
geo_json_layer = folium.GeoJson(
    geojson_data,
    name='Archipelagic Waters',
    style_function=style_function,
    show=True
)
geo_json_layer.add_to(map_archipelagic)

print("✓ Archipelagic waters layer added to map")

🎨 Territory Colors:
   Antigua and Barbuda            -> #e8e407
   Bahamas                        -> #0aae31
   Cape Verde                     -> #7ed7ea
   Chagos Archipelago             -> #5740a4
   Comores                        -> #6ecb5b
✓ Archipelagic waters layer added to map


## Section 8: Customize Map Styling and Interactivity

Add popups, tooltips, and layer controls for enhanced interactivity.

In [8]:
# Create a more advanced layer with popups and tooltips
class ArchipelagicWatersMap:
    def __init__(self, gdf, center_lat, center_lon):
        self.gdf = gdf
        self.map = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=2,
            tiles='OpenStreetMap'
        )
        
    def add_features_with_popups(self):
        """Add features with interactive popups"""
        for idx, row in self.gdf.iterrows():
            # Create popup content
            popup_html = f"""
            <div style="font-family: Arial; width: 300px;">
                <h4 style="margin: 0; color: #003d7a;">{row['GEONAME']}</h4>
                <hr style="margin: 5px 0;">
                <table style="width: 100%; font-size: 12px;">
                    <tr><td><b>Territory:</b></td><td>{row['TERRITORY1']}</td></tr>
                    <tr><td><b>Sovereign:</b></td><td>{row['SOVEREIGN1']}</td></tr>
                    <tr><td><b>Area (km²):</b></td><td>{row['AREA_KM2']:,.0f}</td></tr>
                    <tr><td><b>Type:</b></td><td>{row['POL_TYPE']}</td></tr>
                    <tr><td><b>MRGID:</b></td><td>{row['MRGID']}</td></tr>
                </table>
                <hr style="margin: 5px 0;">
                <small style="color: #666;">
                    Marine Regions v4 | CC-BY License
                </small>
            </div>
            """
            
            # Get centroid for popup
            centroid = row['geometry'].centroid
            
            folium.Marker(
                location=[centroid.y, centroid.x],
                popup=folium.Popup(popup_html, max_width=350),
                icon=folium.Icon(color='blue', icon='info-sign'),
                tooltip=row['GEONAME']
            ).add_to(self.map)
    
    def add_legend(self):
        """Add a legend to the map"""
        legend_html = '''
        <div style="position: fixed; 
                    bottom: 50px; right: 50px; width: 280px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:13px; padding: 15px;
                    border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
            <p style="margin-top: 0; margin-bottom: 10px;"><b>📋 Archipelagic Waters</b></p>
            <p style="margin: 5px 0; font-size: 12px;">
                <i class="fa fa-square" style="color:#1f77b4"></i> Archipelagic water regions
            </p>
            <hr style="margin: 10px 0;">
            <p style="margin: 5px 0; font-size: 11px; color: #666;">
                <b>Data Source:</b> Marine Regions v4<br>
                <b>License:</b> CC-BY 4.0<br>
                <b>Geometry:</b> MultiPolygon<br>
                <b>Features:</b> 23 regions
            </p>
        </div>
        '''
        self.map.get_root().html.add_child(folium.Element(legend_html))
    
    def get_map(self):
        return self.map

# Create the enhanced map
arch_map = ArchipelagicWatersMap(gdf, center_lat, center_lon)
arch_map.add_features_with_popups()
arch_map.add_legend()
map_enhanced = arch_map.get_map()

# Add layer control
folium.LayerControl().add_to(map_enhanced)

print("✓ Interactive features added to map")

✓ Interactive features added to map


## Section 9: Display Final Map

Render the interactive map with all archipelagic waters regions, popups, and controls.

In [10]:
# Display the enhanced map
print("🗺️  Displaying Interactive Map...")
print("   Click on markers to see detailed information about each archipelagic water region")
print("   Use the + and - buttons to zoom in/out")
print("   Drag to pan the map")
print("\n")

# Display the map
map_enhanced

🗺️  Displaying Interactive Map...
   Click on markers to see detailed information about each archipelagic water region
   Use the + and - buttons to zoom in/out
   Drag to pan the map




### Save the Map to HTML File

Export the interactive map to an HTML file for sharing and viewing in web browsers.

In [11]:
# Save the map to HTML
output_path = "/home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/world_archipelagic_waters_map.html"

map_enhanced.save(output_path)

print("✅ Map saved successfully!")
print(f"   File: {output_path}")
print(f"   Size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
print(f"\n🌐 Open in browser: Open the HTML file to view the interactive map")
print(f"   You can also view it at: http://localhost:8000/world_archipelagic_waters_map.html")
print(f"   (if serving from this directory with: python -m http.server 8000)")

✅ Map saved successfully!
   File: /home/crimsondeepdarshak/Desktop/Deep_Darshak/References/Build_1_docs/Fishing_areas/world_archipelagic_waters_map.html
   Size: 0.06 MB

🌐 Open in browser: Open the HTML file to view the interactive map
   You can also view it at: http://localhost:8000/world_archipelagic_waters_map.html
   (if serving from this directory with: python -m http.server 8000)
